In [1]:
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
vectorizer = TfidfVectorizer()
from pathlib import Path
classifier = LogisticRegression(warm_start=True)
import joblib
import numpy as np

In [2]:
def save_model(classifier, vectorizer, model_path):
    joblib.dump(classifier, os.path.join(model_path, 'classifier.pkl'))
    joblib.dump(vectorizer, os.path.join(model_path, 'vectorizer.pkl'))

In [3]:
def load_model(model_path):
    classifier = joblib.load(os.path.join(model_path, 'classifier.pkl'))
    vectorizer = joblib.load(os.path.join(model_path, 'vectorizer.pkl'))
    return classifier, vectorizer

In [4]:
def train_model(path, transfer_learning=False, model_path=None):
    if transfer_learning and model_path:
        old_classifier, old_vectorizer = load_model(model_path)
        vectorizer = old_vectorizer
    else:
        vectorizer = TfidfVectorizer()
        classifier = LogisticRegression(warm_start=True)
    corpus = []
    documents_term_matrixes = []
    authors_per_document = []
    training_path = path
    for file in os.listdir(training_path):
        if (Path(os.path.join(training_path, file))).is_dir():
                for fileName in os.listdir(os.path.join(training_path, file)):
                    with open(os.path.join(training_path, file, fileName), 'r') as f:
                        text = f.read()
                        corpus.append(text.lower())
                        authors_per_document.append(file)
    documents_term_matrixes = vectorizer.fit_transform(corpus)
    if transfer_learning and model_path:
        classifier = LogisticRegression(warm_start=True)
        classifier.coef_ = np.zeros(documents_term_matrixes.shape)
        classifier.intercept_ = old_classifier.intercept_
        classifier.coef_[0, :len(old_classifier.coef_[0])] = old_classifier.coef_[0]
    classifier.fit(documents_term_matrixes, authors_per_document)
    predictions = []
    for document in documents_term_matrixes:
        predictions.append(classifier.predict(document)[0])
    f1 = metrics.f1_score(authors_per_document, predictions, average='macro')
    print(f"F1 Score: {f1}")
    score = classifier.score(documents_term_matrixes, authors_per_document)
    print(f"Accuracy: {score}")
    return classifier, vectorizer

classifier, vectorizer = train_model('./reuter+50+50/C50train/')

F1 Score: 0.9429337610307675
Accuracy: 0.9432


In [5]:
def test_model(classifier, vectorizer, testing_path):
    test_corpus = []
    test_authors = []
    test_matrix = []
    for file in os.listdir(testing_path):
        if (Path(os.path.join(testing_path, file))).is_dir():
            for fileName in os.listdir(os.path.join(testing_path, file)):
                with open(os.path.join(testing_path, file, fileName), 'r') as f:
                    text = f.read()
                    test_corpus.append(text.lower())
                    test_authors.append(file)
    test_matrix = vectorizer.transform(test_corpus)
    # test if this can predict who wrote each document
    predictions = []
    for i, document in enumerate(test_corpus):
        X_test = test_matrix[i]
        predicted_author = classifier.predict(X_test)
        predictions.append(predicted_author[0])
        # print(f"Predicted author for document: {predicted_author[0]} actual: {test_authors[i]}")
    f1 = metrics.f1_score(test_authors, predictions, average='macro')
    print(f"F1 Score: {f1}")
    score = classifier.score(test_matrix, test_authors)
    print(f"Accuracy: {score}")
    
test_model(classifier, vectorizer, './reuter+50+50/C50test/')

F1 Score: 0.6377368899238064
Accuracy: 0.6432


In [6]:
save_model(classifier, vectorizer, './models/')

In [7]:
classifier, vectorizer = load_model('./models/')